In [1]:
%pip install --pre azure-ai-projects azure-identity openai azure-search-documents

In [13]:
import os
import json
import time
import datetime
from pprint import pprint
from pathlib import Path
from typing import Any, List, Dict, Iterable, Union
from packaging.version import Version

import pandas as pd

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, ClientSecretCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import DatasetVersion
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileID,
)

from collections.abc import Mapping
from azure.core.exceptions import ResourceNotFoundError

In [ ]:
AZURE_AI_PROJECT_ENDPOINT = "https://angandin-foundryproject-resource.services.ai.azure.com/api/projects/angandin-foundryproject" # The Azure AI Project project endpoint, as found in the Home page of your Microsoft Foundry portal.

AZURE_TENANT_ID = "3268e587-3ee5-42d0-b9ac-23fcaf66f510"
AZURE_CLIENT_ID = "ce7e72a5-4ed1-40ac-ab78-fbce8f09bd5e"
AZURE_CLIENT_SECRET = \"<REDACTED-SET-VIA-ENV-OR-KEYVAULT>\"

In [4]:
endpoint = AZURE_AI_PROJECT_ENDPOINT

tenant_id = AZURE_TENANT_ID
client_id = AZURE_CLIENT_ID
client_secret = AZURE_CLIENT_SECRET

In [5]:
# using service principal
cred = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)
project_client = AIProjectClient(endpoint=endpoint, credential=cred)

In [6]:
def safe_getattr(obj, path, default=None):
    """
    Access nested attributes via dotted path, e.g. 'data_source.target.name'.
    Returns default if any segment is missing/None.
    """
    cur = obj
    for part in path.split('.'):
        if cur is None:
            return default
        # handle dicts transparently if some layers are dicts
        if isinstance(cur, dict):
            cur = cur.get(part, default)
        else:
            cur = getattr(cur, part, default)
    return cur


In [7]:
client = project_client.get_openai_client()

eval_list_iter = client.evals.list()

evaluation_items = []
run_items = []
run_item_items = []

eval_list = list(eval_list_iter)
for eval_object in eval_list:

    eval_runs_list_iter = client.evals.runs.list(eval_object.id)
    eval_runs_list = list(eval_runs_list_iter)

    evaluation_items.append(eval_object)

    for run in eval_runs_list:

        run_items.append(run)

        output_items = list(
            client.evals.runs.output_items.list(
                run_id=run.id, eval_id=eval_object.id
            )
        )
        #pprint(output_items)
        run_item_items.append(output_items)

In [20]:
run_rows = []
criteria_rows = []

for run in run_items:

    dt_utc = datetime.datetime.utcfromtimestamp(safe_getattr(run, "created_at"))

    run_rows.append({
        "id": safe_getattr(run, "id"),
        "eval_id": safe_getattr(run, "eval_id"),
        "run_name": safe_getattr(run, "name"),
        "created_at": dt_utc,
        "target_type": safe_getattr(run, "data_source.target.type"),
        "target_name": safe_getattr(run, "data_source.target.name"),
        "target_version": safe_getattr(run, "data_source.target.version"),
        "target_model": safe_getattr(run, "data_source.target.model"),
        "data_source_type": safe_getattr(run, "data_source.type"),
        "report_url": safe_getattr(run, "report_url"),
        "status": safe_getattr(run, "status"),
        "created_at": safe_getattr(run, "created_at"),
    })
    
    for crit in run.per_testing_criteria_results:
            criteria_rows.append({
                "evalrun_id": run.id,
                "testing_criteria": crit.testing_criteria,
                "failed": crit.failed,
                "passed": crit.passed
            })


df_runs = pd.DataFrame(run_rows)
df_criteria = pd.DataFrame(criteria_rows)
# display(df_runs)
# display(df_criteria)

In [21]:
run_itemresult_rows = []
run_outputitem_rows = []

for run in run_item_items:
    for evaluation in run:
        run_outputitem_rows.append({
            "id": safe_getattr(evaluation, "id"),
            "output_item_id": safe_getattr(evaluation, "id") + "-" + safe_getattr(evaluation, "run_id"),
            "eval_id": safe_getattr(evaluation, "eval_id"),
            "run_id": safe_getattr(evaluation, "run_id"),
            # "cached_tokens": safe_getattr(evaluation, "usage.cached_tokens"),
            # "completion_tokens": safe_getattr(evaluation, "usage.completion_tokens"),
            # "prompt_tokens": safe_getattr(evaluation, "usage.prompt_tokens"),
            # "total_tokens": safe_getattr(evaluation, "usage.total_tokens"),
            "status": safe_getattr(evaluation, "status"),
            # "query": safe_getattr(evaluation, "datasource_item.query"),
            # "context": safe_getattr(evaluation, "datasource_item.context"),
            # "ground_truth": safe_getattr(evaluation, "datasource_item.ground_truth"),
            # "response": safe_getattr(evaluation, "datasource_item.sample.output_text"),
        })

        for result in evaluation.results:
            run_itemresult_rows.append({
                "output_item_id": safe_getattr(evaluation, "id") + "-" + safe_getattr(evaluation, "run_id"),
                "name": safe_getattr(result, "name"),
                "passed": safe_getattr(result, "passed"),
                "score": safe_getattr(result, "score"),
                "reason": safe_getattr(result, "reason"),
                "threshold": safe_getattr(result, "threshold"),
            })

df_run_output_result_item = pd.DataFrame(run_itemresult_rows)
df_run_output_result_item["score"] = pd.to_numeric(df_run_output_result_item["score"], errors="coerce").fillna(0.0)
df_run_output_result_item["passed"] = df_run_output_result_item["passed"].astype("string").fillna("NA")
df_run_output_result_item["reason"] = df_run_output_result_item["reason"].fillna("")

df_run_output_item = pd.DataFrame(run_outputitem_rows)

#display(df_run_result_item)
#display(df_run_output_item)

In [22]:
import pandas as pd
from deltalake import write_deltalake

delta_table_path = "/lakehouse/default/Tables/dbo/runs"
write_deltalake(delta_table_path, df_runs, mode='overwrite', schema_mode='merge', engine='rust', storage_options={"allow_unsafe_rename": "true"})

delta_table_path = "/lakehouse/default/Tables/dbo/run_criteria"
write_deltalake(delta_table_path, df_criteria, mode='overwrite', schema_mode='merge', engine='rust', storage_options={"allow_unsafe_rename": "true"})

delta_table_path = "/lakehouse/default/Tables/dbo/run_output_item"
write_deltalake(delta_table_path, df_run_output_item, mode='overwrite', schema_mode='merge', engine='rust', storage_options={"allow_unsafe_rename": "true"})

delta_table_path = "/lakehouse/default/Tables/dbo/run_output_result_item"
write_deltalake(delta_table_path, df_run_output_result_item, mode='overwrite', schema_mode='merge', engine='rust', storage_options={"allow_unsafe_rename": "true"})